# Survival and A1C: two analyses on OMOP data

This notebook works through two analyses on an OMOP CDM dataset in BigQuery:

1. **Survival.** Do people with pancreatic cancer (and no diabetes) die sooner
   than people with type 2 diabetes (and no pancreatic cancer)? We build both
   cohorts from vocabulary concept sets, define start, end and censoring dates,
   and compare Kaplan–Meier curves.
2. **A1C.** How do hemoglobin A1C values differ between people with and without
   diabetes?

The data is the **CMS DE-SynPUF** synthetic Medicare dataset in OMOP form, prepared
for this workshop (`synpuf_workshop`). Like all synthetic data it is for learning
the mechanics; don't read the results as clinical findings.

Most steps are a short SQL query that returns a pandas DataFrame, followed by a
little pandas.

## Setup

`CDR` names the dataset every query reads from. On the *All of Us* Researcher
Workbench the same variable comes from `os.environ["WORKSPACE_CDR"]`, so the
queries below work there unchanged.

In [ ]:
# Works around a Workbench image bug that can stop the BigQuery client importing (see bq_env.py).
from bq_env import ensure_bigquery
ensure_bigquery()

try:
    import lifelines  # noqa: F401
except ImportError:
    %pip install -q lifelines

import os
import numpy as np
import pandas as pd
import pandas_gbq
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
CDR = f"{PROJECT}.synpuf_workshop"   # on All of Us: CDR = os.environ["WORKSPACE_CDR"]


def query(sql):
    # Run BigQuery SQL and return the result as a pandas DataFrame.
    return pandas_gbq.read_gbq(sql, project_id=PROJECT, progress_bar_type=None)


def id_list(concepts):
    # "1, 2, 3" from a DataFrame's concept_id column, for SQL IN (...) clauses.
    return ", ".join(str(i) for i in concepts["concept_id"])


# Colors for the two groups, used in every plot.
COLORS = {"Only_DM": "#2a78d6", "Only_PC": "#eb6834"}
LABELS = {"Only_DM": "Diabetes only", "Only_PC": "Pancreatic cancer only"}

print("Querying", CDR)

# Part 1: Survival

## 1.1 Concept sets

A **concept set** is the list of vocabulary concepts that count as a condition.
Rather than typing codes by hand, we start from one *seed* concept and take
everything beneath it in the hierarchy. `concept_ancestor` stores every
(ancestor, descendant) pair, including each concept as its own descendant, so
one join does it. We keep only **standard** concepts, the ones clinical tables
use.

In [ ]:
PC_SEED = 42872399     # Primary adenocarcinoma of pancreas
T2DM_SEED = 201826     # Type 2 diabetes mellitus


def concept_set(seed):
    # Standard concepts at or below `seed` in the vocabulary hierarchy.
    return query(f'''
        SELECT c.concept_id, c.concept_name, c.vocabulary_id
        FROM `{CDR}.concept_ancestor` ca
        JOIN `{CDR}.concept` c ON c.concept_id = ca.descendant_concept_id
        WHERE ca.ancestor_concept_id = {seed}
          AND c.standard_concept = 'S'
        ORDER BY c.concept_name
    ''').drop_duplicates("concept_id")


pc_concepts = concept_set(PC_SEED)
t2dm_concepts = concept_set(T2DM_SEED)
assert PC_SEED in set(pc_concepts.concept_id) and T2DM_SEED in set(t2dm_concepts.concept_id)

print(f"Pancreatic cancer: {len(pc_concepts)} concepts; type 2 diabetes: {len(t2dm_concepts)} concepts")
t2dm_concepts

## 1.2 First diagnosis dates

For each person, the earliest `condition_start_date` of any condition in the set.

In [ ]:
def first_diagnosis(concepts, column):
    # One row per person: the earliest start date of a condition in `concepts`.
    df = query(f'''
        SELECT person_id, MIN(condition_start_date) AS {column}
        FROM `{CDR}.condition_occurrence`
        WHERE condition_concept_id IN ({id_list(concepts)})
        GROUP BY person_id
    ''')
    df[column] = pd.to_datetime(df[column])
    return df


pc_dates = first_diagnosis(pc_concepts, "PC_start_date")
dm_dates = first_diagnosis(t2dm_concepts, "DM_start_date")
print(f"{len(pc_dates):,} people with pancreatic cancer, {len(dm_dates):,} with type 2 diabetes")

## 1.3 Two mutually exclusive groups

Someone with both conditions doesn't belong cleanly to either group, so we label
everyone and keep only the people with exactly one.

In [ ]:
both = pc_dates.merge(dm_dates, on="person_id", how="outer")

both["group"] = np.select(
    [both.PC_start_date.isna(),
     both.DM_start_date.isna(),
     both.DM_start_date <= both.PC_start_date],
    ["Only_DM", "Only_PC", "DM_before_PC"],
    default="PC_before_DM",
)
print(both.group.value_counts().to_string())

exclusive_cohort = both[both.group.isin(["Only_DM", "Only_PC"])].copy()

## 1.4 Sign-of-life date

People who don't die are followed until we stop seeing them. Their **sign-of-life
date** is their last recorded condition, measurement or procedure. Death,
visits, observations and drugs are deliberately left out.

In [ ]:
sol = query(f'''
    SELECT person_id, MAX(event_date) AS SoL_date
    FROM (
      SELECT person_id, condition_start_date AS event_date FROM `{CDR}.condition_occurrence`
      UNION ALL
      SELECT person_id, measurement_date FROM `{CDR}.measurement`
      UNION ALL
      SELECT person_id, procedure_date FROM `{CDR}.procedure_occurrence`
    )
    GROUP BY person_id
''')
sol["SoL_date"] = pd.to_datetime(sol["SoL_date"])

before = len(exclusive_cohort)
exclusive_cohort = exclusive_cohort.merge(sol, on="person_id", how="inner")
print(f"Dropped {before - len(exclusive_cohort):,} people with no sign-of-life date")

## 1.5 Survival variables

- `death`: 1 if the person has a death record, else 0.
- Start: the diagnosis date that defines the person's group.
- End: the death date if they died, otherwise the sign-of-life date.
- `Time_to_event`: days from start to end.

In [ ]:
deaths = query(f'''
    SELECT person_id, MIN(death_date) AS death_date
    FROM `{CDR}.death`
    GROUP BY person_id
''')
deaths["death_date"] = pd.to_datetime(deaths["death_date"])

survival_cohort = exclusive_cohort.merge(deaths, on="person_id", how="left")
survival_cohort["death"] = survival_cohort.death_date.notna().astype(int)
survival_cohort["start_date_for_survival"] = survival_cohort.PC_start_date.fillna(survival_cohort.DM_start_date)
survival_cohort["end_date_for_survival"] = survival_cohort.death_date.fillna(survival_cohort.SoL_date)
survival_cohort["Time_to_event"] = (
    survival_cohort.end_date_for_survival - survival_cohort.start_date_for_survival
).dt.days

survival_cohort = survival_cohort[[
    "person_id", "group", "PC_start_date", "DM_start_date", "SoL_date", "death_date",
    "death", "start_date_for_survival", "end_date_for_survival", "Time_to_event",
]]
survival_cohort.head()

## 1.6 Five-year censoring

We drop intervals that aren't positive: someone diagnosed on their last recorded
day contributes no follow-up time. Then anyone followed past five years is
**censored** at 1,825 days, meaning "alive at five years, outcome unknown after".

SynPUF covers only 2008–2010, so nobody reaches five years here. The step is
kept because longer real-world data needs it.

In [ ]:
FIVE_YEARS = 365 * 5

survival_5y = survival_cohort[survival_cohort.Time_to_event > 0].copy()
n_nonpositive = len(survival_cohort) - len(survival_5y)

over = survival_5y.Time_to_event > FIVE_YEARS
survival_5y.loc[over, "Time_to_event"] = FIVE_YEARS
survival_5y.loc[over, "death"] = 0

print(f"Excluded {n_nonpositive:,} records with non-positive follow-up")
print(f"Censored {over.sum():,} records at five years\n")

summary = survival_5y.groupby("group").agg(
    participants=("person_id", "size"),
    deaths=("death", "sum"),
    median_days=("Time_to_event", "median"),
    min_days=("Time_to_event", "min"),
    max_days=("Time_to_event", "max"),
)
summary["pct_died"] = (100 * summary.deaths / summary.participants).round(1)
summary

### Checks

Assertions that fail loudly if an earlier step went wrong.

In [ ]:
s = survival_5y
assert s.person_id.notna().all()
assert s.person_id.is_unique, "each participant appears once"
assert set(s.group) <= {"Only_DM", "Only_PC"}
assert (s.PC_start_date.notna() ^ s.DM_start_date.notna()).all(), "exactly one diagnosis date"
assert (s.Time_to_event > 0).all() and (s.Time_to_event <= FIVE_YEARS).all()
assert set(s.death) <= {0, 1}
assert (s.loc[s.Time_to_event == FIVE_YEARS, "death"] == 0).all()
for col in ["PC_start_date", "DM_start_date", "SoL_date", "death_date"]:
    assert pd.api.types.is_datetime64_any_dtype(s[col]), col
print("All checks passed.")

## 1.7 Kaplan–Meier curves

A Kaplan–Meier curve estimates the share of a group still alive at each point
in time. Censored people count as at risk until they leave, then drop out without
counting as deaths. Shaded bands are 95% confidence intervals.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
medians = {}

for group in ["Only_DM", "Only_PC"]:
    g = survival_5y[survival_5y.group == group]
    kmf = KaplanMeierFitter(label=f"{LABELS[group]} (n={len(g):,})")
    kmf.fit(g.Time_to_event, event_observed=g.death)
    kmf.plot_survival_function(ax=ax, color=COLORS[group], linewidth=2, ci_alpha=0.15)
    medians[LABELS[group]] = kmf.median_survival_time_
    # Direct label at the end of each curve.
    last_t = kmf.survival_function_.index[-1]
    ax.annotate(LABELS[group], (last_t, kmf.survival_function_.iloc[-1, 0]),
                xytext=(6, 0), textcoords="offset points", va="center", color="#333333")

ax.set_xlabel("Days since diagnosis")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.02)
ax.set_xlim(0, survival_5y.Time_to_event.max() * 1.25)   # room for the curve labels
ax.set_title("Survival after diagnosis")
ax.grid(color="#e5e5e5", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower left", frameon=False)
plt.tight_layout()
plt.show()

for name, m in medians.items():
    print(f"Median survival, {name}: {'not reached' if np.isinf(m) else f'{m:,.0f} days'}")

### Log-rank test

The log-rank test asks whether the two curves differ by more than chance would
explain. (This test wasn't part of the original workshop exercise.)

In [ ]:
dm = survival_5y[survival_5y.group == "Only_DM"]
pc = survival_5y[survival_5y.group == "Only_PC"]
result = logrank_test(dm.Time_to_event, pc.Time_to_event,
                      event_observed_A=dm.death, event_observed_B=pc.death)
print(f"Log-rank test statistic {result.test_statistic:.1f}, p = {result.p_value:.2g}")

# Part 2: A1C by diabetes status

## 2.1 Concept sets with descendants and exclusions

Concept sets built in ATLAS are lists of entries. Each entry says whether to
include the concept's **descendants**, and whether it is **excluded**, meaning
removed from the set together with its descendants. `expand_concept_set` turns
such a list into concrete concept IDs.

In [ ]:
def expand_concept_set(expression, respect_exclusions=True):
    # expression: DataFrame of concept_id, includedescendants, isexcluded.
    # Returns the concepts in the set, joined to their names.
    with_desc = expression.loc[expression.includedescendants, "concept_id"]
    descendants = query(f'''
        SELECT ancestor_concept_id, descendant_concept_id
        FROM `{CDR}.concept_ancestor`
        WHERE ancestor_concept_id IN ({", ".join(map(str, with_desc)) or "NULL"})
    ''') if len(with_desc) else pd.DataFrame(columns=["ancestor_concept_id", "descendant_concept_id"])
    below = descendants.groupby("ancestor_concept_id").descendant_concept_id.apply(set)

    include, exclude = set(), set()
    for row in expression.itertuples():
        ids = {row.concept_id} | (below.get(row.concept_id, set()) if row.includedescendants else set())
        (exclude if row.isexcluded and respect_exclusions else include).update(ids)

    ids = sorted(include - exclude)
    return query(f'''
        SELECT concept_id, concept_name, vocabulary_id, standard_concept
        FROM `{CDR}.concept`
        WHERE concept_id IN ({", ".join(map(str, ids))})
        ORDER BY concept_name
    ''')


def expression(rows):
    return pd.DataFrame(rows, columns=["concept_id", "includedescendants", "isexcluded"])

## 2.2 Who has diabetes?

The diabetes definition combines two concept sets, *complicated* and
*uncomplicated* diabetes. A person counts as diabetic from their earliest
condition in either set (`first_diabetes`).

In [ ]:
complicated = expression([
    (442793, True, False),    # Complication due to diabetes mellitus
])
uncomplicated = expression([
    (4008576, True, False),   # Diabetes mellitus without complication
    (201826, True, False),    # Type 2 diabetes mellitus
    (201254, True, False),    # Type 1 diabetes mellitus
])

diabetes_concepts = pd.concat([
    expand_concept_set(complicated),
    expand_concept_set(uncomplicated),
]).drop_duplicates("concept_id")

all_dm_condition = first_diagnosis(diabetes_concepts, "first_diabetes")
print(f"{len(diabetes_concepts)} diabetes concepts; {len(all_dm_condition):,} people with diabetes")

## 2.3 The A1C concept set

The A1C concept set covers the SNOMED A1C procedure and several LOINC lab codes,
each with descendants. Two CPT codes are listed but marked **excluded**.

The original version of this exercise ignored the exclusion flags. Compare the
two expansions below: with exclusions ignored, the "excluded" CPT codes end up
in the set.

In [ ]:
a1c_expression = expression([
    (4184637, True, False),    # Hemoglobin A1c measurement (SNOMED)
    (2212393, False, True),    # CPT 83037, home-device A1C -- excluded
    (724554, False, True),     # CPT 0384U, kidney panel incl. HbA1c -- excluded
    (3004410, True, False),    # LOINC 4548-4
    (3003309, True, False),    # LOINC 4549-2
    (3005673, True, False),    # LOINC 17856-6
    (40765129, True, False),   # LOINC 62388-4
    (40762352, True, False),   # LOINC 59261-8
])

a1c_concepts = expand_concept_set(a1c_expression)
a1c_ignoring_exclusions = expand_concept_set(a1c_expression, respect_exclusions=False)
print(f"{len(a1c_concepts)} concepts respecting exclusions, "
      f"{len(a1c_ignoring_exclusions)} ignoring them")
a1c_concepts

## 2.4 A1C values per person

We fetch every A1C result, attach each person's `first_diabetes` date, and
summarize per person. Measurements are sorted by date first so that `first_A1C`
really is the earliest result.

In [ ]:
a1c = query(f'''
    SELECT person_id, measurement_date, value_as_number
    FROM `{CDR}.measurement`
    WHERE measurement_concept_id IN ({id_list(a1c_concepts)})
''')
a1c["measurement_date"] = pd.to_datetime(a1c["measurement_date"])
a1c = a1c.merge(all_dm_condition, on="person_id", how="left")

per_person = (
    a1c.sort_values(["person_id", "measurement_date"])
       .groupby("person_id")
       .agg(max_A1C=("value_as_number", "max"),
            min_A1C=("value_as_number", "min"),
            mean_A1C=("value_as_number", "mean"),
            A1C_count=("value_as_number", "size"),
            first_A1C=("value_as_number", "first"),
            first_diabetes=("first_diabetes", "first"))
       .reset_index()
)
per_person["diabetes"] = np.where(per_person.first_diabetes.notna(), "diabetes", "no_diabetes")

print(f"{len(a1c):,} A1C results for {len(per_person):,} people")
per_person.groupby("diabetes")[["mean_A1C", "A1C_count"]].describe().round(2)

## 2.5 Mean A1C by diabetes status

Each person counts once here, at their own mean A1C.

In [ ]:
order = ["no_diabetes", "diabetes"]
data = [per_person.loc[per_person.diabetes == g, "mean_A1C"] for g in order]

fig, ax = plt.subplots(figsize=(6, 5))
box = ax.boxplot(data, patch_artist=True, widths=0.5, medianprops={"color": "#1a1a19", "linewidth": 2})
# Diabetes keeps its blue from the survival plot; no diabetes is neutral gray.
for patch, color in zip(box["boxes"], ["#8a8a85", COLORS["Only_DM"]]):
    patch.set(facecolor=color, alpha=0.35, edgecolor=color)

ax.set_xticks([1, 2], [f"No diabetes\n(n={len(data[0]):,})", f"Diabetes\n(n={len(data[1]):,})"])
ax.set_ylabel("Mean A1C (%)")
ax.set_title("Mean A1C per person, by diabetes status")
ax.grid(axis="y", color="#e5e5e5", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Something to notice

A person counts as "diabetes" if they *ever* had a diabetes diagnosis, even if
it came after some of their A1C tests. Some people in the diabetes group may
have early A1C results in the normal range. Can you find them? How would you
change the analysis so each result is compared with the person's diagnosis
status on the day of the test?